# Démonstration RAG francophone

Ce notebook montre comment mettre en place une chaîne Retrieval-Augmented Generation (RAG) en français avec LangChain et des modèles gratuits hébergés sur Hugging Face. Le flux est le suivant :

1. Charger un fichier texte (par défaut `data/source.txt`).
2. Segmenter le contenu et créer des embeddings en français.
3. Indexer les embeddings dans une base vectorielle locale FAISS.
4. Interroger la base via un modèle de génération multilingue.

> Les cellules utilisent uniquement des ressources locales et des modèles gratuits téléchargés depuis Hugging Face Hub.\
> Adaptez les identifiants de modèle si vous souhaitez utiliser d'autres poids francophones ou multilingues.

## Prérequis

- Python 3.10+ recommandé.
- Installez les dépendances :

```bash
pip install -r requirements.txt
```

Les variables configurables se trouvent dans la cellule suivante.

In [ ]:
from pathlib import Path

# Fichier texte à indexer (modifiez si besoin)
DATA_PATH = Path("data/source.txt")

# Modèle d'embedding multilingue
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Modèle de génération (Seq2Seq multilingue, léger et gratuit)
GENERATION_MODEL = "google/flan-t5-small"

# Dossier de cache pour les poids (optionnel)
HF_HOME = Path.home() / ".cache" / "huggingface"
HF_HOME.mkdir(parents=True, exist_ok=True)


## Chargement et préparation des données

On découpe le texte en documents courts avant de les convertir en vecteurs.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Le fichier {DATA_PATH} est introuvable. Ajoutez votre texte avant de continuer.")

raw_text = DATA_PATH.read_text(encoding="utf-8")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)

splits = text_splitter.split_documents([Document(page_content=raw_text)])
print(f"Nombre de segments : {len(splits)}")


## Création des embeddings et de la base vectorielle

On utilise FAISS pour stocker localement les vecteurs.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = FAISS.from_documents(splits, embedding=embeddings)
retriever = vectorstore.as_retriever()


## Chargement du modèle de génération

`flan-t5-small` est léger et fonctionne en français. Remplacez `GENERATION_MODEL` par un modèle francophone plus grand si vous disposez de ressources suffisantes (par exemple `HuggingFaceH4/zephyr-7b-alpha`).

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain.llms import HuggingFacePipeline

model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL)
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)

gen_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
)

llm = HuggingFacePipeline(pipeline=gen_pipeline)


## Chaîne RAG et requête exemple

Le modèle génère une réponse en s'appuyant sur les passages les plus proches dans la base vectorielle.

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
)

question = "Explique le principe de la RAG en français."
result = qa_chain.invoke({"query": question})

print("Question :", question)
print("Réponse :", result["result"])
print("\nExtraits utilisés :")
for doc in result["source_documents"]:
    print("-", doc.page_content.strip())
